## Imports

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("BigData").config("spark.ui.showConsoleProgress", "false").getOrCreate()


## Customers

In [ ]:
df_customers = spark.read.csv("../data/0_raw/olist_customers_dataset.csv", header=True, inferSchema=True)

print("DF CUSTOMERS")
print("\n\n--- Dataframe preview")
df_customers.show(5)
print("\n\n--- Dataframe schema")
df_customers.printSchema()
print("\n\n---Number of lines :", df_customers.count())

df_customers.write.parquet("../data/1_bronze/customers.parquet", mode="overwrite")
print("\n\n✔ Dataset customers successfully exported in parquet format")

## Sellers

In [ ]:
df_sellers = spark.read.csv("../data/0_raw/olist_sellers_dataset.csv", header=True, inferSchema=True)

print("DF SELLERS")
print("\n\n--- Dataframe preview")
df_sellers.show(5)
print("\n\n--- Dataframe schema")
df_sellers.printSchema()
print("\n\n---Number of lines :", df_sellers.count())

df_sellers.write.parquet("../data/1_bronze/sellers.parquet", mode="overwrite")
print("\n\n✔ Dataset sellers successfully exported in parquet format")

## Geolocation

In [ ]:
df_geo = spark.read.csv("../data/0_raw/olist_geolocation_dataset.csv", header=True, inferSchema=True)

print("DF GEOLOCATION")
print("\n\n--- Dataframe preview")
df_geo.show(5)
print("\n\n--- Dataframe schema")
df_geo.printSchema()
print("\n\n---Number of lines :", df_geo.count())

df_geo.write.parquet("../data/1_bronze/geolocation.parquet", mode="overwrite")
print("\n\n✔ Dataset geolocation successfully exported in parquet format")

## Join keys

In [ ]:
# Loading remaining dataframes
df_items = spark.read.csv("../data/0_raw/olist_order_items_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv("../data/0_raw/olist_order_payments_dataset.csv", header=True, inferSchema=True)
df_reviews = spark.read.csv("../data/0_raw/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("../data/0_raw/olist_orders_dataset.csv", header=True, inferSchema=True)
df_products = spark.read.csv("../data/0_raw/olist_products_dataset.csv", header=True, inferSchema=True)
df_categories = spark.read.csv("../data/0_raw/product_category_name_translation.csv", header=True, inferSchema=True)

# Script to identify common columns in all dataframes
dataframes = {
    "items": df_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "orders": df_orders,
    "products": df_products,
    "categories": df_categories,
    "customers": df_customers,
    "sellers": df_sellers,
    "geolocation": df_geo
}

columns = {}
for name, df in dataframes.items():
    for column in df.columns:
        if column.endswith("zip_code_prefix"):
            key_name = "zip_code_prefix"
            has_prefix = True
        else:
            key_name = column
            has_prefix = False

        if key_name not in columns:
            columns[key_name] = []

        if has_prefix:
            columns[key_name].append(f"{name} ({column})")
        else:
            columns[key_name].append(name)

# Display join keys with associated dataframes
print("JOIN KEYS :")

for column, dfs in sorted(columns.items()):
    if len(dfs) > 1:
        print(f"- {column} : {" - ".join(dfs)}")
